In [1]:
!pip install torch transformers datasets diffusers rouge-score matplotlib pillow
!git clone https://github.com/musfiraumar/MAE-VAE-DDPM-RAG


  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for rouge-score: filename=rouge_score-0.1.2-py3-none-any.whl size=24985 sha256=c35a592fdb7fec279585c8027688a6795fe905bc02f1dee31ae0ad20da4017b6
  Stored in directory: /root/.cache/pip/wheels/44/af/da/5ffc433e2786f0b1a9c6f458d5fb8f611d8eb332387f18698f
Successfully built rouge-score
Cloning into 'MAE-VAE-DDPM-RAG'...
remote: Enumerating objects: 13, done.
remote: Counting objects: 100% (13/13), done.
remote: Compressing objects: 100% (11/11), done.
remote: Total 13 (delta 0), reused 13 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (13/13), 506.70 KiB | 9.05 MiB/s, done.
[Errno 2] No such file or directory: '<repo>'
/content


In [4]:
%cd MAE-VAE-DDPM-RAG

/content/MAE-VAE-DDPM-RAG


In [5]:
from google.colab import drive
drive.mount('/content/drive')

!mkdir -p starter/data/uhat
!unzip -q "/content/drive/MyDrive/uhat_data.zip" -d starter/data/uhat
!ls starter/data/uhat

Mounted at /content/drive
characters_test_set   description.docx	digits_test_set   license.txt
characters_train_set  description.pdf	digits_train_set


In [8]:
cd /content/MAE-VAE-DDPM-RAG/starter


/content/MAE-VAE-DDPM-RAG/starter


In [9]:
!python check_starter.py

README.md: 100% 3.38k/3.38k [00:00<00:00, 5.57MB/s]

data/train-00000-of-00001.parquet: downloading bytes:  83% 86.8M/105M [00:01<00:00, 117MB/s, 6.55MB/s  ] 
data/train-00000-of-00001.parquet: downloading bytes: 100% 104M/104M [00:01<00:00, 72.9MB/s, 9.86MB/s  ]
data/train-00000-of-00001.parquet: reconstructing file: 100% 105M/105M [00:01<00:00, 73.0MB/s, 10.1MB/s  ]

data/test-00000-of-00001.parquet: downloading bytes:  98% 34.0M/34.8M [00:00<00:00, 68.3MB/s,  767kB/s  ]
data/test-00000-of-00001.parquet: downloading bytes: 100% 34.7M/34.7M [00:00<00:00, 43.5MB/s, 3.37MB/s  ]
data/test-00000-of-00001.parquet: reconstructing file: 100% 34.8M/34.8M [00:00<00:00, 43.5MB/s, 3.40MB/s  ]

data/validation-00000-of-00001.parquet: downloading bytes:  48% 16.6M/34.8M [00:00<00:00, 38.3MB/s,  416kB/s  ]
data/validation-00000-of-00001.parquet: downloading bytes: 100% 34.8M/34.8M [00:00<00:00, 53.5MB/s, 3.41MB/s  ]
data/validation-00000-of-00001.parquet: reconstructing file: 100% 34.8M/34.8M [00:0

In [12]:
%%writefile model.py
import torch
import torch.nn as nn
import torch.nn.functional as F
class VAE(nn.Module):
    def __init__(self,d=16):
        super().__init__()
        self.d=d
        self.enc_conv=nn.Sequential(nn.Conv2d(1,32,4,stride=2,padding=1),nn.ReLU(),nn.Conv2d(32,64,4,stride=2,padding=1),nn.ReLU(),nn.Flatten())
        self.fc_mu=nn.Linear(64*7*7,d)
        self.fc_logvar=nn.Linear(64*7*7,d)
        self.dec_fc=nn.Linear(d,64*7*7)
        self.dec_conv=nn.Sequential(nn.ConvTranspose2d(64,32,4,stride=2,padding=1),nn.ReLU(),nn.ConvTranspose2d(32,1,4,stride=2,padding=1))
    def encode(self,x):
        h=self.enc_conv(x)
        return self.fc_mu(h),self.fc_logvar(h)
    def reparameterize(self,mu,logvar):
        std=torch.exp(0.5*logvar)
        eps=torch.randn_like(std)
        return mu+std*eps
    def decode(self,z):
        h=self.dec_fc(z)
        h=h.view(-1,64,7,7)
        return self.dec_conv(h)
    def forward(self,x):
        mu,logvar=self.encode(x)
        z=self.reparameterize(mu,logvar)
        return self.decode(z),mu,logvar
def kl_divergence(mu,logvar):
    return -0.5*torch.sum(1+logvar-mu.pow(2)-logvar.exp(),dim=1)
def elbo_terms(logits,x,mu,logvar):
    recon=F.binary_cross_entropy_with_logits(logits,x,reduction="none").sum(dim=(1,2,3))
    kl=kl_divergence(mu,logvar)
    return recon,kl

Writing model.py


In [15]:

import sys,time,json
import torch
from torch.utils.data import DataLoader,TensorDataset
sys.path.append("../starter")
from uhat import load_uhat_characters
from model import VAE,elbo_terms
torch.manual_seed(0)
device="cuda" if torch.cuda.is_available() else "cpu"
x_tr,l_tr,names=load_uhat_characters(root="../starter/data/uhat",split="train")
x_te,l_te,_=load_uhat_characters(root="../starter/data/uhat",split="test",class_names=names)
print("train",tuple(x_tr.shape),"test",tuple(x_te.shape),"classes",len(names))
train_loader=DataLoader(TensorDataset(x_tr),batch_size=128,shuffle=True)
test_loader=DataLoader(TensorDataset(x_te),batch_size=128,shuffle=False)
model=VAE(d=16).to(device)
optimizer=torch.optim.Adam(model.parameters(),lr=1e-3)
n_params=sum(p.numel() for p in model.parameters())
print("parameters:",n_params)
def run_epoch(loader,train):
    model.train(train)
    sums={"recon":0.0,"kl":0.0,"total":0.0}
    n=0
    with torch.set_grad_enabled(train):
        for (x,) in loader:
            x=x.to(device)
            logits,mu,logvar=model(x)
            recon,kl=elbo_terms(logits,x,mu,logvar)
            loss=(recon+kl).mean()
            if train:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
            sums["recon"]+=recon.sum().item()
            sums["kl"]+=kl.sum().item()
            sums["total"]+=(recon+kl).sum().item()
            n+=x.size(0)
    return {k:v/n for k,v in sums.items()}
log={"train":[],"test":[]}
start=time.time()
for epoch in range(1,31):
    tr=run_epoch(train_loader,train=True)
    te=run_epoch(test_loader,train=False)
    log["train"].append(tr)
    log["test"].append(te)
    print(f"epoch {epoch:2d} | train recon {tr['recon']:.2f} kl {tr['kl']:.2f} total {tr['total']:.2f} | test recon {te['recon']:.2f} kl {te['kl']:.2f} total {te['total']:.2f}")
elapsed=time.time()-start
log["params"]=n_params
log["seconds"]=elapsed
log["gpu"]=torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu"
torch.save(model.state_dict(),"../results/vae.pt")
with open("../results/vae_log.json","w") as f:
    json.dump(log,f,indent=2)
print("done in",round(elapsed),"seconds")

train (28328, 1, 28, 28) test (4747, 1, 28, 28) classes 40
parameters: 220385
epoch  1 | train recon 186.25 kl 14.61 total 200.86 | test recon 141.89 kl 18.61 total 160.50
epoch  2 | train recon 137.41 kl 19.05 total 156.46 | test recon 132.32 kl 18.94 total 151.26
epoch  3 | train recon 130.04 kl 19.98 total 150.01 | test recon 127.13 kl 19.73 total 146.86
epoch  4 | train recon 125.69 kl 20.64 total 146.33 | test recon 122.53 kl 21.33 total 143.86
epoch  5 | train recon 122.97 kl 20.97 total 143.94 | test recon 121.94 kl 20.55 total 142.49
epoch  6 | train recon 121.12 kl 21.29 total 142.42 | test recon 120.55 kl 20.51 total 141.06
epoch  7 | train recon 119.71 kl 21.49 total 141.20 | test recon 118.78 kl 21.11 total 139.89
epoch  8 | train recon 118.47 kl 21.79 total 140.26 | test recon 117.49 kl 21.84 total 139.34
epoch  9 | train recon 117.40 kl 22.05 total 139.45 | test recon 117.34 kl 21.21 total 138.55
epoch 10 | train recon 116.45 kl 22.22 total 138.67 | test recon 115.42 kl 2

In [17]:
!cp /content/MAE-VAE-DDPM-RAG/results/vae.pt "/content/drive/MyDrive/vae.pt"

In [19]:
import sys,json,os
import torch
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
sys.path.append("/content/MAE-VAE-DDPM-RAG/starter")
from uhat import load_uhat_characters
from viz import save_grid
from model import VAE
torch.manual_seed(0)
device="cuda" if torch.cuda.is_available() else "cpu"
os.makedirs("/content/MAE-VAE-DDPM-RAG/results",exist_ok=True)
_,_,names=load_uhat_characters(root="/content/MAE-VAE-DDPM-RAG/starter/data/uhat",split="train")
x_te,l_te,_=load_uhat_characters(root="/content/MAE-VAE-DDPM-RAG/starter/data/uhat",split="test",class_names=names)
model=VAE(16).to(device)
model.load_state_dict(torch.load("/content/MAE-VAE-DDPM-RAG/results/vae.pt",map_location=device))
model.eval()
log=json.load(open("/content/MAE-VAE-DDPM-RAG/results/vae_log.json"))
epochs=range(1,len(log["train"])+1)
fig,axes=plt.subplots(1,3,figsize=(15,4))
for ax,key,title in zip(axes,["recon","kl","total"],["Reconstruction (BCE)","KL","Negative ELBO"]):
    ax.plot(epochs,[e[key] for e in log["train"]],label="train")
    ax.plot(epochs,[e[key] for e in log["test"]],label="test")
    ax.set_title(title)
    ax.set_xlabel("epoch")
    ax.set_ylabel("nats per image")
    ax.legend()
fig.tight_layout()
fig.savefig("/content/MAE-VAE-DDPM-RAG/results/vae_loss_terms.png",dpi=120)
plt.close(fig)
with torch.no_grad():
    idx=torch.randperm(len(x_te))[:16]
    x=x_te[idx].to(device)
    mu,logvar=model.encode(x)
    recon=torch.sigmoid(model.decode(mu))
    save_grid(torch.cat([x,recon]),"/content/MAE-VAE-DDPM-RAG/results/vae_reconstructions.png",nrow=16)
    z=torch.randn(64,16,device=device)
    samples=torch.sigmoid(model.decode(z))
    save_grid(samples,"/content/MAE-VAE-DDPM-RAG/results/vae_samples.png",nrow=8)
    i=0
    j=int((l_te!=l_te[i]).nonzero()[0])
    pair=x_te[[i,j]].to(device)
    mu_pair,_=model.encode(pair)
    alphas=torch.linspace(0,1,10,device=device).view(10,1)
    z_line=(1-alphas)*mu_pair[0]+alphas*mu_pair[1]
    interp=torch.sigmoid(model.decode(z_line))
    save_grid(interp,"/content/MAE-VAE-DDPM-RAG/results/vae_interpolation.png",nrow=10)
    print("interpolating",names[int(l_te[i])],"->",names[int(l_te[j])])
print("final test:",log["test"][-1])

interpolating hamza -> alif
final test: {'recon': 105.62005241077424, 'kl': 25.044204155821685, 'total': 130.66425449008915}


In [20]:
import torch
from model import VAE,kl_divergence
mu0,lv0=torch.zeros(8,16),torch.zeros(8,16)
print("KL at mu=0, logvar=0:",kl_divergence(mu0,lv0))
assert torch.allclose(kl_divergence(mu0,lv0),torch.zeros(8))
mu=torch.randn(10000,16)*3
lv=torch.randn(10000,16)*2
kl=kl_divergence(mu,lv)
print("min KL over 10000 random cases:",kl.min().item())
assert (kl>=-1e-5).all()
m=VAE(16)
x=torch.rand(4,1,28,28)
logits,mu_,lv_=m(x)
print("logits",tuple(logits.shape),"mu",tuple(mu_.shape),"logvar",tuple(lv_.shape))

KL at mu=0, logvar=0: tensor([-0., -0., -0., -0., -0., -0., -0., -0.])
min KL over 10000 random cases: 24.49400520324707
logits (4, 1, 28, 28) mu (4, 16) logvar (4, 16)
